# Leakage-Safe Linguistic Branch — Final v11 Revision (Kaggle)

**Purpose:** produce the corrected linguistic branch for the later audio–linguistic fusion, using the same `DEPRESSION_DATASET` structure as the acoustic notebook.

This notebook preserves the v11 psycholinguistic feature definitions and the leakage-safe LOO/model-selection protocol. The main change is dataset handling: it reads directly from the Kaggle-mounted folders rather than Google Drive ZIP files.

### Kaggle dataset layout expected
```text
/kaggle/input/datasets/schizospeech/depression-diac-woz/DEPRESSION_DATASET/
├── Train_Audios/
├── Train_Transcripts/
├── Test_Audios/
├── Test_Transcripts/
├── train_split_Depression_AVEC2017.csv
├── test_split_Depression_AVEC2017.csv
└── full_test_split.csv
```

For this linguistic branch, `Train_Transcripts` + `train_split_Depression_AVEC2017.csv` are used for model development. `Test_Transcripts` + `full_test_split.csv` are used only for the final held-out evaluation.

**Important:** do not use the old Google Drive ZIP files or old v11 probabilities for the corrected fusion.

## 1. Install dependencies

In [1]:
!pip install -q transformers torch scikit-learn pandas numpy tqdm vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.8 MB/s eta 0:00:0000:01


## 2. Imports and configuration

In [2]:
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    classification_report, confusion_matrix,
    accuracy_score, f1_score, brier_score_loss
)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

# -------------------- KAGGLE DATASET PATHS --------------------
# These paths intentionally mirror the uploaded acoustic notebook.
DATASET_ROOT = "/kaggle/input/datasets/schizospeech/depression-diac-woz/DEPRESSION_DATASET"
TRAIN_TRANSCRIPTS_DIR = os.path.join(DATASET_ROOT, "Train_Transcripts")
TEST_TRANSCRIPTS_DIR  = os.path.join(DATASET_ROOT, "Test_Transcripts")
TRAIN_CSV = os.path.join(DATASET_ROOT, "train_split_Depression_AVEC2017.csv")
TEST_CSV  = os.path.join(DATASET_ROOT, "full_test_split.csv")

# Kaggle keeps generated files under /kaggle/working.
MODEL_SAVE = "/kaggle/working/depression_model_v11_final_leakage_safe.pkl"
OOF_SAVE   = "/kaggle/working/linguistic_oof_predictions_v11_final.csv"

RUN_TEST_EVAL = True

# -------------------- MODEL SETTINGS --------------------
SEED = 42
N_SVD = 20
C_GRID = [0.005, 0.01, 0.02, 0.05, 0.1]
INNER_SPLITS = 5
CALIBRATION_SPLITS = 5

BERT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
CHUNK_SIZE = 510
STRIDE = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("ClinicalBERT:", BERT_MODEL)
print("Dataset root:", DATASET_ROOT)
print("Train transcripts:", TRAIN_TRANSCRIPTS_DIR)
print("Test transcripts:", TEST_TRANSCRIPTS_DIR)
print("Train CSV exists:", os.path.isfile(TRAIN_CSV))
print("Test CSV exists:", os.path.isfile(TEST_CSV))
print("SVD components:", N_SVD)
print("C grid:", C_GRID)


Device: cuda
ClinicalBERT: emilyalsentzer/Bio_ClinicalBERT
Dataset root: /kaggle/input/datasets/schizospeech/depression-diac-woz/DEPRESSION_DATASET
Train transcripts: /kaggle/input/datasets/schizospeech/depression-diac-woz/DEPRESSION_DATASET/Train_Transcripts
Test transcripts: /kaggle/input/datasets/schizospeech/depression-diac-woz/DEPRESSION_DATASET/Test_Transcripts
Train CSV exists: True
Test CSV exists: True
SVD components: 20
C grid: [0.005, 0.01, 0.02, 0.05, 0.1]


## 2.5. Dataset structure check

This section intentionally follows the same folder structure used by the acoustic notebook. It does not merge audio and transcript files; the linguistic branch only reads the transcript folders and the official split CSVs.

The training split is the 107-participant development partition. The held-out evaluation uses `full_test_split.csv` with `Test_Transcripts/`.

## 3. Load DAIC-WOZ training transcripts and labels

In [3]:
# ── Dataset audit and transcript loading ─────────────────────────
assert os.path.isdir(TRAIN_TRANSCRIPTS_DIR), f"Missing: {TRAIN_TRANSCRIPTS_DIR}"
assert os.path.isfile(TRAIN_CSV), f"Missing: {TRAIN_CSV}"

def list_transcript_files(directory):
    files = []
    for root, _, filenames in os.walk(directory):
        for name in filenames:
            if name.lower().endswith(".csv"):
                files.append(os.path.join(root, name))
    return sorted(files)

def detect_id_column(columns):
    for col in columns:
        c = str(col).strip().lower().replace(" ", "_")
        if c in {"participant_id", "participant", "pid", "id", "subject"}:
            return col
    for col in columns:
        if "participant" in str(col).lower():
            return col
    raise KeyError(f"Could not identify participant-ID column from: {list(columns)}")

def detect_label_column(columns):
    candidates = {"phq8_binary", "phq_binary", "phq8binary", "phqbinary", "label", "depression"}
    for col in columns:
        norm = str(col).strip().lower().replace(" ", "").replace("_", "")
        if norm in {c.replace("_", "") for c in candidates}:
            return col
    return None

def detect_score_column(columns):
    for col in columns:
        norm = str(col).strip().lower().replace(" ", "").replace("_", "")
        if norm in {"phq8score", "phqscore", "phq_score"}:
            return col
    return None

def parse_time(t):
    try:
        return float(str(t).strip())
    except Exception:
        return None

NOISE = re.compile(r'<[^>]+>|\[[^\]]+\]|\([^)]+\)', re.IGNORECASE)

def clean(text):
    return re.sub(r'\s+', ' ', NOISE.sub(' ', str(text))).strip()

def load_dialogue(fpath):
    try:
        df = pd.read_csv(
            fpath, sep='\t', header=0,
            names=['start_time','stop_time','speaker','value'],
            dtype=str, on_bad_lines='skip'
        )
        turns = []
        for _, row in df.iterrows():
            spk = str(row['speaker']).strip().lower()
            txt = clean(str(row['value']))
            start = parse_time(row['start_time'])
            stop = parse_time(row['stop_time'])
            if len(txt) < 2 or not re.search(r'[a-zA-Z]', txt):
                continue
            if spk == 'participant':
                turns.append({'speaker': 'participant', 'text': txt,
                              'start': start, 'stop': stop})
            elif spk in ('ellie', 'interviewer', 'therapist', 'clinician'):
                turns.append({'speaker': 'interviewer', 'text': txt,
                              'start': start, 'stop': stop})
        return turns if turns else None
    except Exception:
        return None

def find_transcript(pid, files):
    try:
        pid_int = int(float(pid))
    except Exception:
        return None
    candidates = []
    for fpath in files:
        nums = re.findall(r'\d+', os.path.basename(fpath))
        if nums:
            try:
                if int(nums[0]) == pid_int:
                    candidates.append(fpath)
            except Exception:
                pass
    return sorted(candidates)[0] if candidates else None

train_transcript_files = list_transcript_files(TRAIN_TRANSCRIPTS_DIR)
train_meta = pd.read_csv(TRAIN_CSV)
train_meta.columns = train_meta.columns.str.strip()

TRAIN_ID_COL = detect_id_column(train_meta.columns)
TRAIN_LABEL_COL = detect_label_column(train_meta.columns)
TRAIN_SCORE_COL = detect_score_column(train_meta.columns)

print(f"Train transcript files found: {len(train_transcript_files)}")
print(f"Train CSV rows: {len(train_meta)}")
print(f"Participant ID column: {TRAIN_ID_COL}")
print(f"Binary label column: {TRAIN_LABEL_COL}")
print(f"PHQ-8 score column: {TRAIN_SCORE_COL}")
assert TRAIN_LABEL_COL is not None, "Training CSV must contain a PHQ binary label column."
assert TRAIN_SCORE_COL is not None, "Training CSV must contain a PHQ-8 score column."

records = []
missing_transcripts = []
unreadable_transcripts = []

for _, row in train_meta.iterrows():
    pid = row[TRAIN_ID_COL]
    fpath = find_transcript(pid, train_transcript_files)
    if not fpath:
        missing_transcripts.append(pid)
        continue
    turns = load_dialogue(fpath)
    if not turns:
        unreadable_transcripts.append(pid)
        continue
    phq_score = row[TRAIN_SCORE_COL]
    phq_binary = row[TRAIN_LABEL_COL]
    if pd.isna(phq_score) or pd.isna(phq_binary):
        continue
    patient_text = " ".join(t['text'] for t in turns if t['speaker'] == 'participant')
    if len(patient_text.strip()) < 10:
        unreadable_transcripts.append(pid)
        continue
    records.append({
        "pid": pid,
        "turns": turns,
        "patient_text": patient_text,
        "phq_score": float(phq_score),
        "phq_binary": int(phq_binary),
    })

df_meta = pd.DataFrame(records)
N = len(records)

print("\n" + "="*65)
print("TRAIN TRANSCRIPT MATCHING SUMMARY")
print("="*65)
print(f"CSV rows:                 {len(train_meta)}")
print(f"Transcript files found:   {len(train_transcript_files)}")
print(f"Matched usable records:   {N}")
print(f"Missing transcript IDs:   {len(missing_transcripts)}")
print(f"Unreadable/short records: {len(unreadable_transcripts)}")
print(f"Depressed:                {int(df_meta['phq_binary'].sum()) if N else 0}")
print(f"Non-depressed:            {int((df_meta['phq_binary']==0).sum()) if N else 0}")
print("="*65)

if missing_transcripts:
    print("Missing transcript IDs:", missing_transcripts)
if unreadable_transcripts:
    print("Unreadable/short IDs:", unreadable_transcripts)

assert N >= 100, f"Expected approximately 107 training participants; only {N} were loaded. Fix dataset matching before continuing."


Train transcript files found: 107
Train CSV rows: 107
Participant ID column: Participant_ID
Binary label column: PHQ8_Binary
PHQ-8 score column: PHQ8_Score

TRAIN TRANSCRIPT MATCHING SUMMARY
CSV rows:                 107
Transcript files found:   107
Matched usable records:   107
Missing transcript IDs:   0
Unreadable/short records: 0
Depressed:                30
Non-depressed:            77


## 4. Load frozen ClinicalBERT

ClinicalBERT is used only as a pretrained representation extractor. Its parameters are not fine-tuned on DAIC-WOZ.

**Kaggle note:** the first run needs access to the Hugging Face checkpoint `emilyalsentzer/Bio_ClinicalBERT`. If Kaggle Internet is disabled and the model is not already cached/attached, enable Internet or attach the model files as a Kaggle dataset. The checkpoint is fixed; there is no silent fallback to another model.

In [4]:
# ── ClinicalBERT ───────────────────────────────────────────────
print("\nLoading ClinicalBERT...")
BERT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"

# No silent fallback is used in the final revision.
# The exact checkpoint must remain fixed for reproducibility.
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE)

print(f"Loaded exact checkpoint: {BERT_MODEL}")

for p in bert.parameters():
    p.requires_grad = False
bert.eval()

@torch.no_grad()
def embed_full_text(text, chunk_size=510, stride=256):
    token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if not token_ids: return np.zeros(768, dtype=np.float32)
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id or 0
    chunk_embs, start = [], 0
    while start < len(token_ids):
        end     = min(start + chunk_size, len(token_ids))
        chunk   = [cls_id] + token_ids[start:end] + [sep_id]
        pad_len = (chunk_size + 2) - len(chunk)
        ids     = chunk + [pad_id] * pad_len
        mask    = [1] * len(chunk) + [0] * pad_len
        out = bert(input_ids=torch.tensor([ids]).to(DEVICE),
                   attention_mask=torch.tensor([mask]).to(DEVICE))
        chunk_embs.append(
            out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy())
        if end == len(token_ids): break
        start += stride
    return np.mean(chunk_embs, axis=0).astype(np.float32)

print("Embedding participant speech...")
bert_embs = np.array([embed_full_text(r["patient_text"])
                      for r in tqdm(records)])
print(f"BERT embeddings: {bert_embs.shape}")


Loading ClinicalBERT...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (2 outputs left out)

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded exact checkpoint: emilyalsentzer/Bio_ClinicalBERT
Embedding participant speech...


100%|██████████| 107/107 [00:18<00:00,  5.72it/s]

BERT embeddings: (107, 768)


## 5. Original v11 psycholinguistic feature extraction

This cell preserves the v11 feature definitions and lexicons. The revision is focused on making the **learning/evaluation pipeline** leakage-safe rather than silently changing the submitted feature space.


In [5]:
# ── Lexicons ───────────────────────────────────────────────────
vader = SentimentIntensityAnalyzer()

SADNESS = {
    "sad","sorrow","sorrowful","unhappy","miserable","grief","grieve",
    "grieving","cry","crying","tears","weep","weeping","desolate",
    "hopeless","despair","despairing","desperate","heartbroken","broken",
    "empty","hollow","numb","dark","darkness","bleak","gloomy","gloom",
    "depressed","depression","down","low","lost","alone","lonely",
    "loneliness","isolated","isolation","worthless","useless","failure",
    "failed","helpless","trapped","stuck","meaningless","pointless",
    "nothing","nobody","never","always","forever","anymore"
}
ANHEDONIA = {
    "enjoy","enjoyed","enjoyment","pleasure","fun","interest","interested",
    "interesting","hobby","hobbies","motivation","motivated","excited",
    "excitement","care","caring","want","wanted","desire","passion",
    "passionate","engage","engaged","engagement","social","friends",
    "family","activities","activity","used to"
}
SOMATIC = {
    "sleep","sleeping","slept","insomnia","tired","tiredness","exhausted",
    "exhaustion","fatigue","fatigued","energy","energetic","appetite",
    "eating","weight","concentration","concentrate","focus","focused",
    "distracted","distraction","slow","sluggish","heavy","ache","aching",
    "pain","painful","headache","stomach","body","physical","restless",
    "restlessness","agitated","agitation"
}
COGNITIVE = {
    "think","thinking","thought","thoughts","mind","mental","forget",
    "forgetting","forgetful","memory","remember","remembering","confused",
    "confusion","unclear","difficult","difficulty","hard","struggle",
    "struggling","overwhelmed","overwhelming","worry","worrying","worried",
    "ruminate","ruminating","rumination","doubt","doubting","uncertain",
    "uncertainty","blame","blaming","regret","regretting","should",
    "must","never","always","everyone","no one","nothing","everything"
}
POS_AFFECT = {
    "good","great","happy","wonderful","love","enjoy","fine","okay",
    "well","nice","better","best","exciting","positive","grateful",
    "comfortable","relaxed","fortunate","lucky","appreciate","glad",
    "hope","hopeful","motivated","energetic","calm","peaceful","content",
    "pleased","satisfied","proud","confident","strong","healthy"
}
DISFLUENCY  = {"um","uh","uhh","umm","hmm","mhm","mm","uh-huh","hm"}
FP_WORDS    = {"i","me","my","myself","mine"}
HEDGES      = {
    "maybe","perhaps","possibly","probably","might","could","sometimes",
    "kind","sort","roughly","approximately","around","about","somewhat",
    "rather","fairly","like","i think","i guess","i suppose","i believe",
}
CERTAINTY   = {
    "definitely","certainly","absolutely","clearly","obviously","always",
    "never","every","all","none","must","will","know","sure","certain",
    "exactly","completely","totally","entirely","really","truly"
}
FUNCTION_WORDS = {
    "a","an","the","is","are","was","were","be","been","being",
    "have","has","had","do","does","did","will","would","could","should",
    "may","might","shall","can","and","but","or","nor","for","yet","so",
    "at","by","for","from","in","into","of","on","to","with","i","me",
    "my","we","our","you","your","he","she","it","they","them","their",
    "this","that","these","those","not","no","if","as","than","then",
    "when","while","where","who","which","what","how","so","just","also"
}
SUBORD_CONJ = {
    "because","although","though","even","while","since","if","unless",
    "until","when","whenever","where","wherever","whether","as","after",
    "before","once","so","than","that","despite","however","whereas"
}

PHQ_DOMAINS = [
    {"little","interest","pleasure","enjoy"},
    {"feeling","down","depressed","hopeless"},
    {"sleep","sleeping","slept","insomnia","oversleep"},
    {"tired","energy","fatigue","exhausted"},
    {"appetite","eating","weight","overeating"},
    {"bad","failure","guilty","disappointment","let","down"},
    {"concentrate","concentration","focus","distracted"},
    {"slow","moving","speaking","restless","fidgety"},
]
OPEN_Q_MARKERS = {
    "tell","describe","explain","how","what","why","when","where","talk"}

def is_open_question(text):
    first = text.lower().split()[0] if text.split() else ""
    return first in OPEN_Q_MARKERS

def turn_ttr(text):
    w = text.lower().split()
    return len(set(w)) / len(w) if len(w) >= 2 else 1.0

def turn_filler_ratio(text):
    w = text.lower().split()
    return sum(x in DISFLUENCY for x in w) / len(w) if w else 0.0

def linreg_slope(values):
    n = len(values)
    if n < 2: return 0.0
    x = np.arange(n, dtype=np.float32) - (n - 1) / 2.0
    y = np.array(values, dtype=np.float32) - np.mean(values)
    denom = np.dot(x, x)
    return float(np.dot(x, y) / denom) if denom > 1e-9 else 0.0

def approx_syllables(word):
    return max(1, len(re.findall(r'[aeiouAEIOU]+', word)))

def extract_features(rec):
    turns        = rec["turns"]
    patient_text = rec["patient_text"]
    words        = patient_text.lower().split()
    sentences    = [s.strip() for s in re.split(r'[.!?]+', patient_text)
                    if s.strip()]
    n_words  = max(len(words), 1)
    n_sents  = max(len(sentences), 1)
    sent_lens = [len(s.split()) for s in sentences] or [0]

    pat_turns = [t for t in turns if t['speaker'] == 'participant']
    int_turns = [t for t in turns if t['speaker'] == 'interviewer']
    n_pat = max(len(pat_turns), 1)
    n_int = max(len(int_turns), 1)

    ttr          = len(set(words)) / n_words
    avg_word_len = float(np.mean([len(w) for w in words])) if words else 0.0
    avg_sent_len = n_words / n_sents
    sent_std     = float(np.std(sent_lens))

    sadness_count   = sum(w in SADNESS   for w in words)
    anhedonia_count = sum(w in ANHEDONIA for w in words)
    somatic_count   = sum(w in SOMATIC   for w in words)
    cognitive_count = sum(w in COGNITIVE for w in words)
    pos_count       = sum(w in POS_AFFECT  for w in words)
    disf_count      = sum(w in DISFLUENCY  for w in words)
    fp_count        = sum(w in FP_WORDS    for w in words)

    sadness_r    = sadness_count   / n_words
    anhedonia_r  = anhedonia_count / n_words
    somatic_r    = somatic_count   / n_words
    cognitive_r  = cognitive_count / n_words
    pos_r        = pos_count       / n_words
    disf_r       = disf_count      / n_words
    fp_r         = fp_count        / n_words

    neg_total    = sadness_count + somatic_count + cognitive_count
    neg_r        = neg_total / n_words
    affect_ratio = neg_r / max(pos_r, 0.001)
    domain_div   = (int(sadness_count > 0) + int(somatic_count > 0) +
                    int(cognitive_count > 0) + int(anhedonia_count > 0))

    log_wc        = np.log1p(n_words)
    log_neg       = np.log1p(neg_total)
    short_ratio   = sum(l <= 4 for l in sent_lens) / n_sents
    sent_density  = n_sents / max(log_wc, 1)
    neg_sent_ratio = sum(
        any(w in SADNESS or w in SOMATIC or w in COGNITIVE
            for w in s.lower().split())
        for s in sentences) / n_sents

    mild_sig     = np.clip(neg_total /  5.0, 0, 1)
    moderate_sig = np.clip(neg_total / 10.0, 0, 1)
    severe_sig   = np.clip(neg_total / 15.0, 0, 1)

    wc_dict = {}
    for w in words:
        wc_dict[w] = wc_dict.get(w, 0) + 1
    repetition = sum(v > 2 for v in wc_dict.values()) / max(len(wc_dict), 1)
    q_ratio    = patient_text.count('?') / n_sents

    pat_texts = [t['text'] for t in pat_turns]

    turn_ttrs     = [turn_ttr(t) for t in pat_texts]
    mean_turn_ttr = float(np.mean(turn_ttrs))
    std_turn_ttr  = float(np.std(turn_ttrs)) if len(turn_ttrs) > 1 else 0.0
    min_turn_ttr  = float(np.min(turn_ttrs))

    turn_sent_vars = []
    for t in pat_texts:
        st = [s.strip() for s in re.split(r'[.!?]+', t) if s.strip()]
        lt = [len(s.split()) for s in st] if st else [len(t.split())]
        turn_sent_vars.append(float(np.var(lt)))
    mean_sent_var = float(np.mean(turn_sent_vars))

    turn_fillers = [turn_filler_ratio(t) for t in pat_texts]
    mean_filler  = float(np.mean(turn_fillers))
    std_filler   = float(np.std(turn_fillers)) if len(turn_fillers) > 1 else 0.0

    seen_vocab = set(); novelties = []
    for t in pat_texts:
        tw = set(t.lower().split())
        novelties.append(len(tw - seen_vocab) / max(len(tw), 1))
        seen_vocab |= tw
    mean_novelty = float(np.mean(novelties)) if novelties else 1.0

    n_pairs = max(len(pat_texts) - 1, 1)
    monotony_count = 0
    for i in range(1, len(pat_texts)):
        pw = set(pat_texts[i-1].lower().split())
        cw = set(pat_texts[i].lower().split())
        if pw and cw and len(pw & cw) / len(pw | cw) > 0.5:
            monotony_count += 1
    monotony_idx = monotony_count / n_pairs

    window = 5
    moving_ttrs = []
    if len(pat_texts) >= window:
        for i in range(len(pat_texts) - window + 1):
            cw = " ".join(pat_texts[i:i+window]).lower().split()
            if cw: moving_ttrs.append(len(set(cw)) / len(cw))
    if moving_ttrs:
        mover_ttr_mean  = float(np.mean(moving_ttrs))
        mover_ttr_min   = float(np.min(moving_ttrs))
        mover_ttr_trend = float(moving_ttrs[-1] - moving_ttrs[0])
    else:
        mover_ttr_mean  = mean_turn_ttr
        mover_ttr_min   = min_turn_ttr
        mover_ttr_trend = 0.0

    flatness_score = ((1.0 - mean_turn_ttr) * 0.4 +
                      mean_filler            * 0.3 +
                      (1.0 - mean_novelty)  * 0.3)

    vader_scores = [vader.polarity_scores(t)['compound'] for t in pat_texts]
    if vader_scores:
        vader_mean  = float(np.mean(vader_scores))
        vader_std   = float(np.std(vader_scores))
        vader_min   = float(np.min(vader_scores))
        vader_max   = float(np.max(vader_scores))
        third_v     = max(len(vader_scores) // 3, 1)
        vader_early = float(np.mean(vader_scores[:third_v]))
        vader_late  = float(np.mean(vader_scores[-third_v:]))
        vader_trend = vader_late - vader_early
        neg_turn_r  = sum(s < -0.3 for s in vader_scores) / len(vader_scores)
        pos_turn_r  = sum(s >  0.3 for s in vader_scores) / len(vader_scores)
        vader_range = vader_max - vader_min
    else:
        vader_mean = vader_std = vader_min = vader_max = 0.0
        vader_early = vader_late = vader_trend = 0.0
        neg_turn_r = pos_turn_r = vader_range = 0.0

    latencies, speaking_rates, syllable_rates = [], [], []
    for i, t in enumerate(turns):
        if t['speaker'] == 'participant':
            n_w = len(t['text'].split())
            if t.get('start') is not None and t.get('stop') is not None:
                dur = t['stop'] - t['start']
                if dur > 0.1:
                    speaking_rates.append(n_w / dur)
                    n_syl = sum(approx_syllables(w)
                                for w in t['text'].lower().split())
                    syllable_rates.append(n_syl / dur)
            if i > 0 and turns[i-1]['speaker'] == 'interviewer':
                ps = turns[i-1].get('stop'); cs = t.get('start')
                if ps is not None and cs is not None:
                    lat = cs - ps
                    if 0 <= lat <= 30:
                        latencies.append(lat)

    avg_latency       = float(np.mean(latencies))      if latencies         else -1.0
    std_latency       = float(np.std(latencies))       if len(latencies)>1  else 0.0
    avg_spk_rate      = float(np.mean(speaking_rates)) if speaking_rates    else -1.0
    std_spk_rate      = float(np.std(speaking_rates))  if len(speaking_rates)>1 else 0.0
    long_pause_r      = sum(l > 3.0 for l in latencies) / max(len(latencies), 1)
    avg_syl_rate      = float(np.mean(syllable_rates)) if syllable_rates    else -1.0
    pause_variability = std_latency
    lat_trend         = linreg_slope(latencies) if len(latencies) >= 3 else 0.0
    spk_rate_trend    = linreg_slope(speaking_rates) if len(speaking_rates)>=3 else 0.0
    word_pause_var    = std_spk_rate

    pat_turn_lens = [len(t['text'].split()) for t in pat_turns]
    avg_pat_len   = float(np.mean(pat_turn_lens))
    std_pat_len   = float(np.std(pat_turn_lens)) if len(pat_turn_lens)>1 else 0.0
    short_turns   = sum(l <= 5  for l in pat_turn_lens) / n_pat
    long_turns    = sum(l >  50 for l in pat_turn_lens) / n_pat
    int_turn_lens = [len(t['text'].split()) for t in int_turns]
    total_pat_wds = sum(pat_turn_lens)
    total_int_wds = sum(int_turn_lens) if int_turn_lens else 1
    talk_ratio    = total_pat_wds / max(total_pat_wds + total_int_wds, 1)

    elaborations, open_elabs, closed_elabs = [], [], []
    elab_seq = []
    for i, t in enumerate(turns):
        if t['speaker'] == 'participant' and i > 0:
            prev = turns[i-1]
            if prev['speaker'] == 'interviewer':
                ratio = len(t['text'].split()) / max(len(prev['text'].split()), 1)
                elaborations.append(ratio)
                elab_seq.append(ratio)
                (open_elabs if is_open_question(prev['text'])
                 else closed_elabs).append(ratio)

    if elaborations:
        avg_elab = float(np.mean(elaborations))
        std_elab = float(np.std(elaborations)) if len(elaborations)>1 else 0.0
        q25_elab = float(np.percentile(elaborations, 25))
        q75_elab = float(np.percentile(elaborations, 75))
        iqr_elab = q75_elab - q25_elab
        min_elab = float(np.min(elaborations))
    else:
        avg_elab = std_elab = q25_elab = q75_elab = iqr_elab = min_elab = 1.0

    avg_open_elab     = float(np.mean(open_elabs))   if open_elabs   else 1.0
    avg_closed_elab   = float(np.mean(closed_elabs)) if closed_elabs else 1.0
    open_closed_ratio = avg_open_elab / max(avg_closed_elab, 0.1)
    open_q_frac       = sum(is_open_question(t['text'])
                            for t in int_turns) / max(n_int, 1)
    elaboration_decline = linreg_slope(elab_seq) if len(elab_seq) >= 3 else 0.0

    third     = max(len(words) // 3, 1)
    neg_early = sum(w in SADNESS or w in SOMATIC
                    for w in words[:third]) / third
    neg_late  = sum(w in SADNESS or w in SOMATIC
                    for w in words[-third:]) / third
    neg_trend = neg_late - neg_early

    def turn_neg_density(ts):
        aw = " ".join(t['text'] for t in ts).lower().split()
        return sum(w in SADNESS for w in aw) / max(len(aw), 1)

    q_sz   = max(n_pat // 4, 1)
    neg_q1 = turn_neg_density(pat_turns[:q_sz])
    neg_q2 = turn_neg_density(pat_turns[q_sz:2*q_sz])
    neg_q3 = turn_neg_density(pat_turns[2*q_sz:3*q_sz])
    neg_q4 = turn_neg_density(pat_turns[3*q_sz:])
    neg_q_range = (max(neg_q1, neg_q2, neg_q3, neg_q4) -
                   min(neg_q1, neg_q2, neg_q3, neg_q4))

    phq_hits     = sum(any(w in d for w in words) for d in PHQ_DOMAINS)
    phq_coverage = phq_hits / 8.0

    subord_count = sum(w in SUBORD_CONJ for w in words)
    clause_ratio = subord_count / n_sents

    long_sents_list = [l for l in sent_lens if l > 10]
    sentence_complexity = float(np.mean(long_sents_list)) if long_sents_list else 0.0

    content_word_count = sum(1 for w in words if w not in FUNCTION_WORDS)
    lexical_density = content_word_count / n_words

    hedge_count     = sum(1 for w in words if w in HEDGES)
    certainty_count = sum(1 for w in words if w in CERTAINTY)
    hedging_ratio   = hedge_count     / n_words
    certainty_ratio = certainty_count / n_words

    turn_len_trend = linreg_slope(pat_turn_lens) if len(pat_turn_lens) >= 3 else 0.0

    repeat_count = 0
    for i, t in enumerate(turns):
        if t['speaker'] == 'interviewer' and i > 0:
            prev = turns[i-1]
            if prev['speaker'] == 'participant':
                pat_w = set(prev['text'].lower().split()) - FUNCTION_WORDS
                int_w = set(t['text'].lower().split()) - FUNCTION_WORDS
                if pat_w and int_w:
                    overlap = len(pat_w & int_w) / max(len(pat_w), 1)
                    if overlap > 0.2:
                        repeat_count += 1
    int_repeat_rate = repeat_count / max(n_int, 1)

    return np.array([
        ttr, avg_word_len, avg_sent_len, sent_std,
        sadness_r, anhedonia_r, somatic_r, cognitive_r, pos_r,
        disf_r, fp_r, affect_ratio,
        log_wc, log_neg, short_ratio, sent_density, neg_sent_ratio,
        repetition, q_ratio,
        mild_sig, moderate_sig, severe_sig,
        sadness_r * fp_r, domain_div / 4.0, neg_r * fp_r,
        vader_mean, vader_std, vader_min, vader_max,
        vader_early, vader_late, vader_trend,
        neg_turn_r, pos_turn_r, vader_mean * fp_r, vader_range,
        avg_latency, std_latency, avg_spk_rate, std_spk_rate, long_pause_r,
        avg_syl_rate, pause_variability, lat_trend, spk_rate_trend, word_pause_var,
        avg_pat_len, std_pat_len, short_turns, long_turns, talk_ratio,
        float(n_pat), float(n_int),
        float(n_pat) / max(float(len(turns)), 1),
        phq_coverage, open_q_frac,
        avg_elab, std_elab, q25_elab, q75_elab, iqr_elab, min_elab,
        avg_open_elab, avg_closed_elab, open_closed_ratio,
        elaboration_decline,
        neg_q1, neg_q2, neg_q3, neg_q4, neg_q_range,
        neg_early, neg_late, neg_trend,
        np.log1p(sadness_count), np.log1p(somatic_count),
        np.log1p(cognitive_count), np.log1p(anhedonia_count),
        np.log1p(disf_count), np.log1p(fp_count), np.log1p(n_words),
        mean_turn_ttr, std_turn_ttr, min_turn_ttr,
        mean_sent_var, mean_filler, std_filler,
        mean_novelty, monotony_idx,
        mover_ttr_mean, mover_ttr_min, mover_ttr_trend,
        flatness_score, vader_range * (1.0 - mean_turn_ttr),
        clause_ratio, sentence_complexity, lexical_density,
        hedging_ratio, certainty_ratio,
        turn_len_trend, int_repeat_rate,
    ], dtype=np.float32)

feat_names_short = (
    ["ttr","avg_wl","avg_sl","sent_std",
     "sad_r","anh_r","som_r","cog_r","pos_r","disf_r","fp_r","aff_r",
     "log_wc","log_neg","shr_r","snt_den","neg_sr","rep","q_r",
     "mild","mod","sev","sad*fp","dom_div","neg*fp",
     "vdr_mn","vdr_sd","vdr_mn2","vdr_mx","vdr_e","vdr_l","vdr_tr",
     "neg_tr","pos_tr","vdr*fp","vdr_rng",
     "avg_lat","std_lat","avg_spk","std_spk","lng_pse",
     "avg_syl","pause_var","lat_trend","spk_rate_trend","word_pause_var",
     "avg_pl","std_pl","sht_tr","lng_tr","tlk_r","n_pat","n_int",
     "pt_frac","phq_cov","opq_frac",
     "avg_el","std_el","q25_el","q75_el","iqr_el","min_el",
     "op_el","cl_el","oc_r","elab_dec",
     "neg_q1","neg_q2","neg_q3","neg_q4","neg_qr",
     "neg_e","neg_l","neg_tr2",
     "l_sad","l_som","l_cog","l_anh","l_dis","l_fp","l_nw",
     "mtt_mn","mtt_sd","mtt_mn2","m_sv","m_fil","s_fil",
     "m_nov","mon_i","mvt_mn","mvt_mi","mvt_tr","flat","flat*v",
     "clause_r","sent_complex","lex_den","hedge_r","cert_r",
     "tl_trend","int_rep"]
)

## 6. Extract embeddings, psycholinguistic features, and targets

In [6]:
print("Embedding participant speech...")
bert_embs = np.array([
    embed_full_text(r["patient_text"], chunk_size=CHUNK_SIZE, stride=STRIDE)
    for r in tqdm(records)
])
print("BERT embeddings:", bert_embs.shape)

print("\nExtracting psycholinguistic features...")
rich_feats = np.array([extract_features(r) for r in tqdm(records)])
print("Rich features:", rich_feats.shape)

y_bin = df_meta["phq_binary"].values.astype(np.int64)
y_scr = df_meta["phq_score"].values.astype(np.float32)
pids = df_meta["pid"].values

assert len(rich_feats) == len(bert_embs) == len(y_bin)

RICH_DIM = rich_feats.shape[1]
BERT_DIM = bert_embs.shape[1]

print(f"Participants: {len(y_bin)}")
print(f"Depressed: {int(y_bin.sum())}")
print(f"Non-depressed: {int((y_bin == 0).sum())}")
print(f"Psycholinguistic dimensions: {RICH_DIM}")
print(f"ClinicalBERT dimensions: {BERT_DIM}")


Embedding participant speech...


100%|██████████| 107/107 [00:16<00:00,  6.32it/s]


BERT embeddings: (107, 768)

Extracting psycholinguistic features...


100%|██████████| 107/107 [00:01<00:00, 54.13it/s]

Rich features: (107, 101)
Participants: 107
Depressed: 30
Non-depressed: 77
Psycholinguistic dimensions: 101
ClinicalBERT dimensions: 768


## 7. Leakage-safe preprocessing, model selection, and calibration

Every learned transformation is placed inside the model pipeline.

For each outer LOO fold:

1. rich-feature variance filtering is fit only on the outer training data;
2. rich-feature scaling is fit only on the outer training data;
3. ClinicalBERT SVD is fit only on the outer training data;
4. BERT scaling is fit only on the outer training data;
5. C is selected by inner CV using only the outer training data;
6. sigmoid/Platt calibration is fit only inside the outer training data;
7. the held-out participant is predicted once.

This is the main methodological correction needed before the linguistic probabilities are passed to fusion.


In [7]:
def make_feature_matrix(X_rich, X_bert):
    return np.concatenate([X_rich, X_bert], axis=1)


def build_base_pipeline(C_value):
    rich_indices = list(range(RICH_DIM))
    bert_indices = list(range(RICH_DIM, RICH_DIM + BERT_DIM))

    rich_pipe = Pipeline([
        ("variance_filter", VarianceThreshold(threshold=1e-4)),
        ("scaler", StandardScaler()),
    ])

    bert_pipe = Pipeline([
        ("svd", TruncatedSVD(n_components=N_SVD, random_state=SEED)),
        ("scaler", StandardScaler()),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("rich", rich_pipe, rich_indices),
            ("bert", bert_pipe, bert_indices),
        ],
        remainder="drop"
    )

    classifier = LogisticRegression(
        C=C_value,
        class_weight="balanced",
        max_iter=2000,
        solver="lbfgs",
        random_state=SEED
    )

    return Pipeline([
        ("preprocess", preprocessor),
        ("classifier", classifier)
    ])


def make_calibrated_model(C_value, calibration_splits=CALIBRATION_SPLITS):
    base = build_base_pipeline(C_value)

    try:
        return CalibratedClassifierCV(
            estimator=base,
            method="sigmoid",
            cv=calibration_splits,
            ensemble=True
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base,
            method="sigmoid",
            cv=calibration_splits,
            ensemble=True
        )


def select_C_inner(X_raw, y_train):
    """Select C using only the current outer-training partition."""
    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    results = []

    for C_value in C_GRID:
        model = build_base_pipeline(C_value)

        try:
            scores = cross_val_score(
                model,
                X_raw,
                y_train,
                cv=inner_cv,
                scoring="roc_auc",
                n_jobs=1,
                error_score="raise"
            )
            mean_auc = float(np.mean(scores))
        except Exception as exc:
            print(f"  C={C_value}: inner CV failed -> {exc}")
            mean_auc = -np.inf

        results.append((C_value, mean_auc))
        print(f"  C={C_value:0.4f} | inner ROC-AUC={mean_auc:.4f}")

    return max(results, key=lambda x: x[1])[0], max(results, key=lambda x: x[1])[1], results


## 8. Build the combined raw feature matrix

In [8]:
X_raw = make_feature_matrix(rich_feats, bert_embs)
print("Raw combined matrix:", X_raw.shape)


Raw combined matrix: (107, 869)


## 9. Leakage-safe Leave-One-Out linguistic predictions

These are the predictions that should later become the **linguistic OOF input to fusion**.

For each participant, every learned component is trained without that participant.


In [9]:
loo_probs = np.zeros(len(y_bin), dtype=np.float64)
loo_best_C = np.zeros(len(y_bin), dtype=np.float64)
loo_inner_auc = np.zeros(len(y_bin), dtype=np.float64)

loo = LeaveOneOut()

for fold, (tr_idx, val_idx) in enumerate(
    tqdm(loo.split(X_raw), total=len(y_bin), desc="Leakage-safe LOO")
):
    tr_idx = np.asarray(tr_idx)
    val_idx = np.asarray(val_idx)

    X_tr = X_raw[tr_idx]
    y_tr = y_bin[tr_idx]
    X_val = X_raw[val_idx]

    best_C, best_inner_auc, _ = select_C_inner(X_tr, y_tr)

    loo_best_C[val_idx[0]] = best_C
    loo_inner_auc[val_idx[0]] = best_inner_auc

    calibrated_model = make_calibrated_model(best_C)
    calibrated_model.fit(X_tr, y_tr)

    loo_probs[val_idx[0]] = calibrated_model.predict_proba(X_val)[0, 1]

print("\nLeakage-safe LOO predictions generated.")
print("OOF probability range:", float(loo_probs.min()), "to", float(loo_probs.max()))


Leakage-safe LOO:   0%|          | 0/107 [00:00<?, ?it/s]

  C=0.0050 | inner ROC-AUC=0.7175
  C=0.0100 | inner ROC-AUC=0.7242
  C=0.0200 | inner ROC-AUC=0.7176
  C=0.0500 | inner ROC-AUC=0.6961
  C=0.1000 | inner ROC-AUC=0.6806


Leakage-safe LOO:   1%|          | 1/107 [00:05<09:27,  5.35s/it]

  C=0.0050 | inner ROC-AUC=0.7199
  C=0.0100 | inner ROC-AUC=0.7244
  C=0.0200 | inner ROC-AUC=0.7157
  C=0.0500 | inner ROC-AUC=0.6875
  C=0.1000 | inner ROC-AUC=0.6810


Leakage-safe LOO:   2%|▏         | 2/107 [00:12<10:45,  6.15s/it]

  C=0.0050 | inner ROC-AUC=0.7111
  C=0.0100 | inner ROC-AUC=0.7112
  C=0.0200 | inner ROC-AUC=0.7089
  C=0.0500 | inner ROC-AUC=0.6915
  C=0.1000 | inner ROC-AUC=0.6829


... (206 outputs left out)

Leakage-safe LOO:  99%|█████████▉| 106/107 [09:04<00:05,  5.52s/it]

  C=0.0050 | inner ROC-AUC=0.7200
  C=0.0100 | inner ROC-AUC=0.7269
  C=0.0200 | inner ROC-AUC=0.7290
  C=0.0500 | inner ROC-AUC=0.6937
  C=0.1000 | inner ROC-AUC=0.6803


Leakage-safe LOO: 100%|██████████| 107/107 [09:09<00:00,  5.13s/it]


Leakage-safe LOO predictions generated.
OOF probability range: 0.11111957857286799 to 0.6087788356692028


## 10. LOO metrics, training-derived threshold, and calibration diagnostics

In [10]:
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2/(2*n)) / denom
    half = z * np.sqrt((p*(1-p) + z**2/(4*n))/n) / denom
    return center - half, center + half


def classification_metrics(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) else np.nan
    npv  = tn / (tn + fn) if (tn + fn) else np.nan

    return {
        "threshold": threshold,
        "auc": roc_auc_score(y_true, prob),
        "accuracy": accuracy_score(y_true, pred),
        "precision_ppv": ppv,
        "sensitivity": sens,
        "specificity": spec,
        "npv": npv,
        "f1": f1_score(y_true, pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "sensitivity_ci": wilson_ci(tp, tp + fn),
        "specificity_ci": wilson_ci(tn, tn + fp),
        "ppv_ci": wilson_ci(tp, tp + fp),
        "npv_ci": wilson_ci(tn, tn + fn),
    }


loo_auc = roc_auc_score(y_bin, loo_probs)

# Training-only threshold selection.
fpr, tpr, thresholds = roc_curve(y_bin, loo_probs)
youden = tpr - fpr
best_idx = int(np.argmax(youden))
loo_youden_threshold = float(thresholds[best_idx])

loo_metrics = classification_metrics(
    y_bin, loo_probs, loo_youden_threshold
)

print(f"Leakage-safe LOO ROC-AUC: {loo_auc:.4f}")
print(f"Training-derived Youden threshold: {loo_youden_threshold:.4f}")

print("\nLOO metrics:")
for k, v in loo_metrics.items():
    if not k.endswith("_ci"):
        print(f"  {k}: {v}")

print("\n95% Wilson CIs:")
print("  Sensitivity:", loo_metrics["sensitivity_ci"])
print("  Specificity:", loo_metrics["specificity_ci"])
print("  PPV:", loo_metrics["ppv_ci"])
print("  NPV:", loo_metrics["npv_ci"])

brier = brier_score_loss(y_bin, loo_probs)
print(f"\nOOF Brier score: {brier:.4f}")

prob_true, prob_pred = calibration_curve(
    y_bin, loo_probs, n_bins=5, strategy="quantile"
)

calibration_table = pd.DataFrame({
    "mean_predicted_probability": prob_pred,
    "observed_fraction_positive": prob_true
})
display(calibration_table)

print("\nSelected C frequency:")
print(pd.Series(loo_best_C).value_counts().sort_index())


Leakage-safe LOO ROC-AUC: 0.7091
Training-derived Youden threshold: 0.4294

LOO metrics:
  threshold: 0.42939829640827565
  auc: 0.7090909090909091
  accuracy: 0.7663551401869159
  precision_ppv: 0.5925925925925926
  sensitivity: 0.5333333333333333
  specificity: 0.8571428571428571
  npv: 0.825
  f1: 0.5614035087719298
  tn: 66
  fp: 11
  fn: 14
  tp: 16

95% Wilson CIs:
  Sensitivity: (np.float64(0.3614201328152505), np.float64(0.6976787277587178))
  Specificity: (np.float64(0.7620252720484026), np.float64(0.9183175217607253))
  PPV: (np.float64(0.4072638305536409), np.float64(0.7548548662714265))
  NPV: (np.float64(0.7274225612441251), np.float64(0.8927946101839012))

OOF Brier score: 0.1774


,mean_predicted_probability,observed_fraction_positive
0,0.169207,0.227273
1,0.228872,0.095238
2,0.285714,0.190476
3,0.386224,0.238095
4,0.512325,0.636364



Selected C frequency:
0.005    22
0.010    31
0.020    54
Name: count, dtype: int64


## 11. Save leakage-safe OOF predictions for fusion

**This CSV is the hand-off file for the later fusion notebook.**

Do not replace these probabilities with in-sample predictions from the final model.


In [11]:
oof_df = pd.DataFrame({
    "Participant_ID": pids,
    "PHQ8_Binary": y_bin,
    "PHQ8_Score": y_scr,
    "linguistic_oof_probability": loo_probs,
    "selected_C": loo_best_C,
    "inner_cv_auc": loo_inner_auc
})

oof_df.to_csv(OOF_SAVE, index=False)

print("Saved:", OOF_SAVE)
display(oof_df.head())


Saved: /kaggle/working/linguistic_oof_predictions_v11_final.csv


,Participant_ID,PHQ8_Binary,PHQ8_Score,linguistic_oof_probability,selected_C,inner_cv_auc
0,303.0,0,0.0,0.291469,0.01,0.724167
1,304.0,0,6.0,0.315657,0.01,0.724444
2,305.0,0,7.0,0.434902,0.01,0.711250
3,310.0,0,4.0,0.281819,0.02,0.713750
4,312.0,0,2.0,0.259676,0.02,0.713750


## 12. Select final C using training data only

In [12]:
final_C, final_inner_auc, final_C_results = select_C_inner(X_raw, y_bin)

print(f"Final selected C: {final_C}")
print(f"Final inner-CV ROC-AUC: {final_inner_auc:.4f}")


  C=0.0050 | inner ROC-AUC=0.7249
  C=0.0100 | inner ROC-AUC=0.7336
  C=0.0200 | inner ROC-AUC=0.7360
  C=0.0500 | inner ROC-AUC=0.7122
  C=0.1000 | inner ROC-AUC=0.6949
Final selected C: 0.02
Final inner-CV ROC-AUC: 0.7360


## 13. Train the final calibrated linguistic model

This is the model used for the held-out test set and later test-side fusion inference.

No test labels are used.


In [13]:
final_model = make_calibrated_model(final_C)
final_model.fit(X_raw, y_bin)
print("Final calibrated linguistic model trained.")


Final calibrated linguistic model trained.


## 14. Optional held-out test evaluation

The test set is used **only for final evaluation**.

The threshold is the Youden threshold selected from training-set LOO predictions. Test labels do not affect calibration, C selection, threshold selection, or model fitting.


In [14]:
def load_records_from_folder(transcript_dir, meta_path, require_labels=True):
    assert os.path.isdir(transcript_dir), f"Missing transcript directory: {transcript_dir}"
    assert os.path.isfile(meta_path), f"Missing metadata CSV: {meta_path}"

    transcript_files = list_transcript_files(transcript_dir)
    meta_local = pd.read_csv(meta_path)
    meta_local.columns = meta_local.columns.str.strip()

    id_col = detect_id_column(meta_local.columns)
    label_col = detect_label_column(meta_local.columns)
    score_col = detect_score_column(meta_local.columns)

    if require_labels:
        assert label_col is not None and score_col is not None, \
            f"Test CSV must contain PHQ label and score columns. Columns: {list(meta_local.columns)}"

    records_local = []
    missing = []

    for _, row in meta_local.iterrows():
        pid = row[id_col]
        fpath = find_transcript(pid, transcript_files)
        if not fpath:
            missing.append(pid)
            continue

        turns = load_dialogue(fpath)
        if not turns:
            continue

        patient_text = " ".join(
            t["text"] for t in turns if t["speaker"] == "participant"
        )
        if len(patient_text.strip()) < 10:
            continue

        rec = {
            "pid": pid,
            "turns": turns,
            "patient_text": patient_text,
        }

        if label_col is not None:
            if pd.isna(row[label_col]):
                continue
            rec["phq_binary"] = int(row[label_col])
        if score_col is not None:
            if pd.isna(row[score_col]):
                continue
            rec["phq_score"] = float(row[score_col])

        records_local.append(rec)

    print(f"Transcript files: {len(transcript_files)}")
    print(f"CSV rows: {len(meta_local)}")
    print(f"Usable records: {len(records_local)}")
    print(f"Missing transcript IDs: {len(missing)}")
    if missing:
        print("Missing IDs:", missing)
    return records_local


def evaluate_test_set():
    if not os.path.isdir(TEST_TRANSCRIPTS_DIR) or not os.path.isfile(TEST_CSV):
        print("Held-out test files not found; skipping test evaluation.")
        return None

    test_records = load_records_from_folder(
        TEST_TRANSCRIPTS_DIR, TEST_CSV, require_labels=True
    )

    print(f"\nTest records loaded: {len(test_records)}")
    if len(test_records) == 0:
        return None

    test_embs = np.array([
        embed_full_text(r["patient_text"], chunk_size=CHUNK_SIZE, stride=STRIDE)
        for r in tqdm(test_records, desc="Test ClinicalBERT")
    ])

    test_rich = np.array([
        extract_features(r)
        for r in tqdm(test_records, desc="Test psycholinguistic features")
    ])

    test_y = np.array([r["phq_binary"] for r in test_records], dtype=np.int64)
    test_ids = np.array([r["pid"] for r in test_records])
    test_X = make_feature_matrix(test_rich, test_embs)
    test_prob = final_model.predict_proba(test_X)[:, 1]

    test_auc = roc_auc_score(test_y, test_prob)
    test_metrics = classification_metrics(
        test_y, test_prob, loo_youden_threshold
    )

    print("\n" + "="*70)
    print("HELD-OUT TEST — LINGUISTIC BRANCH")
    print("="*70)
    print(f"N = {len(test_y)}")
    print(f"ROC-AUC = {test_auc:.4f}")
    print(f"Training LOO Youden threshold = {loo_youden_threshold:.4f}")
    print(f"Sensitivity = {test_metrics['sensitivity']:.4f}")
    print(f"Specificity = {test_metrics['specificity']:.4f}")
    print(f"PPV = {test_metrics['precision_ppv']:.4f}")
    print(f"NPV = {test_metrics['npv']:.4f}")
    print(f"F1 = {test_metrics['f1']:.4f}")
    print(f"Accuracy = {test_metrics['accuracy']:.4f}")
    print("="*70)

    return {
        "ids": test_ids,
        "y": test_y,
        "prob": test_prob,
        "auc": test_auc,
        "metrics": test_metrics
    }


test_results = evaluate_test_set() if RUN_TEST_EVAL else None


Transcript files: 46
CSV rows: 47
Usable records: 46
Missing transcript IDs: 1
Missing IDs: [np.int64(334)]

Test records loaded: 46


Test psycholinguistic features: 100%|██████████| 46/46 [00:01<00:00, 45.26it/s]



HELD-OUT TEST — LINGUISTIC BRANCH
N = 46
ROC-AUC = 0.8036
Training LOO Youden threshold = 0.4294
Sensitivity = 0.2143
Specificity = 0.9062
PPV = 0.5000
NPV = 0.7250
F1 = 0.3000
Accuracy = 0.6957


## 15. Save the final linguistic bundle

The bundle is intentionally designed for the later fusion stage:

- `oof_predictions` contains the leakage-safe training predictions;
- `model` is the final calibrated linguistic model for held-out/test inference;
- the exact ClinicalBERT checkpoint and chunking settings are stored;
- the training-derived threshold is stored;
- no isotonic severity model is included.


In [15]:
bundle = {
    "model": final_model,
    "bert_model": BERT_MODEL,
    "chunk_size": CHUNK_SIZE,
    "stride": STRIDE,
    "n_svd": N_SVD,
    "final_C": final_C,
    "rich_dim": RICH_DIM,
    "bert_dim": BERT_DIM,
    "threshold_youden_loo": loo_youden_threshold,
    "auc_loo": loo_auc,
    "brier_loo": brier,
    "feature_names": list(feat_names_short),
    "oof_predictions": oof_df.copy(),
    "test_results": test_results,
    "c_grid": C_GRID,
    "inner_splits": INNER_SPLITS,
    "calibration_splits": CALIBRATION_SPLITS,
    "seed": SEED,
    "dataset_root": DATASET_ROOT,
    "train_csv": TRAIN_CSV,
    "test_csv": TEST_CSV,
}

with open(MODEL_SAVE, "wb") as f:
    pickle.dump(bundle, f)

print("Saved final linguistic bundle ->", MODEL_SAVE)
print("Saved OOF predictions ->", OOF_SAVE)


Saved final linguistic bundle -> /kaggle/working/depression_model_v11_final_leakage_safe.pkl
Saved OOF predictions -> /kaggle/working/linguistic_oof_predictions_v11_final.csv


## 16. Final hand-off audit

Before running the corrected fusion notebook, confirm:

- [ ] The dataset audit shows approximately 107 usable training participants.
- [ ] `linguistic_oof_predictions_v11_final.csv` contains exactly one OOF probability per usable training participant.
- [ ] Every OOF probability was generated without using that participant for preprocessing, C selection, fitting, or calibration.
- [ ] SVD is fitted inside the outer training fold.
- [ ] variance filtering is fitted inside the outer training fold.
- [ ] scaling is fitted inside the outer training fold.
- [ ] C is selected using only the outer training partition.
- [ ] calibration is fitted only within the outer training partition.
- [ ] the held-out test set is not used for any model decision.
- [ ] the test threshold is the training-derived LOO Youden threshold.
- [ ] no old `depression_model_v11.pkl` or old linguistic probabilities are used in fusion.
- [ ] PHQ-8 severity calibration is handled later by the corrected fusion pipeline, not by this linguistic classifier.

**Fusion hand-off:** use `linguistic_oof_probability` from the generated CSV for the 107 training participants. For the 46 held-out test participants, use `test_results['prob']` from the final model (or regenerate the test probabilities from the saved final bundle).